In [1]:
import pandas as pd

df = pd.read_csv("data/used_device_data.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

(3454, 15)
['device_brand', 'os', 'screen_size', '4g', '5g', 'rear_camera_mp', 'front_camera_mp', 'internal_memory', 'ram', 'battery', 'weight', 'release_year', 'days_used', 'normalized_used_price', 'normalized_new_price']


,device_brand,os,screen_size,4g,5g,rear_camera_mp,front_camera_mp,internal_memory,ram,battery,weight,release_year,days_used,normalized_used_price,normalized_new_price
0,Honor,Android,14.50,yes,no,13.0,5.0,64.0,3.0,3020.0,146.0,2020,127,4.307572,4.715100
1,Honor,Android,17.30,yes,yes,13.0,16.0,128.0,8.0,4300.0,213.0,2020,325,5.162097,5.519018
2,Honor,Android,16.69,yes,yes,13.0,8.0,128.0,8.0,4200.0,213.0,2020,162,5.111084,5.884631
3,Honor,Android,25.50,yes,yes,13.0,8.0,64.0,6.0,7250.0,480.0,2020,345,5.135387,5.630961
4,Honor,Android,15.32,yes,no,13.0,8.0,64.0,3.0,5000.0,185.0,2020,293,4.389995,4.947837


In [2]:
# 결측치 확인
print(df.isnull().sum())
print()
# 전체 컬럼 확인 (잘려서 안 보이는 것까지)
print(df.columns.tolist())

device_brand               0
os                         0
screen_size                0
4g                         0
5g                         0
rear_camera_mp           179
front_camera_mp            2
internal_memory            4
ram                        4
battery                    6
weight                     7
release_year               0
days_used                  0
normalized_used_price      0
normalized_new_price       0
dtype: int64

['device_brand', 'os', 'screen_size', '4g', '5g', 'rear_camera_mp', 'front_camera_mp', 'internal_memory', 'ram', 'battery', 'weight', 'release_year', 'days_used', 'normalized_used_price', 'normalized_new_price']


In [3]:
# 숫자형 컬럼 결측치를 중앙값으로 채우기
num_cols = ['rear_camera_mp', 'front_camera_mp', 'internal_memory', 'ram', 'battery', 'weight']

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# 다시 확인
print(df.isnull().sum())

device_brand             0
os                       0
screen_size              0
4g                       0
5g                       0
rear_camera_mp           0
front_camera_mp          0
internal_memory          0
ram                      0
battery                  0
weight                   0
release_year             0
days_used                0
normalized_used_price    0
normalized_new_price     0
dtype: int64


In [4]:
# 문자형(범주형) 컬럼 확인
print(df.dtypes)

device_brand                 str
os                           str
screen_size              float64
4g                           str
5g                           str
rear_camera_mp           float64
front_camera_mp          float64
internal_memory          float64
ram                      float64
battery                  float64
weight                   float64
release_year               int64
days_used                  int64
normalized_used_price    float64
normalized_new_price     float64
dtype: object


In [5]:
# 범주형 컬럼을 원-핫 인코딩으로 변환
categorical_cols = ['device_brand', 'os', '4g', '5g']

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(df_encoded.shape)
df_encoded.head()

(3454, 49)


,screen_size,rear_camera_mp,front_camera_mp,internal_memory,ram,battery,weight,release_year,days_used,normalized_used_price,...,device_brand_Spice,device_brand_Vivo,device_brand_XOLO,device_brand_Xiaomi,device_brand_ZTE,os_Others,os_Windows,os_iOS,4g_yes,5g_yes
0,14.50,13.0,5.0,64.0,3.0,3020.0,146.0,2020,127,4.307572,...,False,False,False,False,False,False,False,False,True,False
1,17.30,13.0,16.0,128.0,8.0,4300.0,213.0,2020,325,5.162097,...,False,False,False,False,False,False,False,False,True,True
2,16.69,13.0,8.0,128.0,8.0,4200.0,213.0,2020,162,5.111084,...,False,False,False,False,False,False,False,False,True,True
3,25.50,13.0,8.0,64.0,6.0,7250.0,480.0,2020,345,5.135387,...,False,False,False,False,False,False,False,False,True,True
4,15.32,13.0,8.0,64.0,3.0,5000.0,185.0,2020,293,4.389995,...,False,False,False,False,False,False,False,False,True,False


In [6]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns=['normalized_used_price'])
y = df_encoded['normalized_used_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("학습 데이터:", X_train.shape)
print("테스트 데이터:", X_test.shape)

학습 데이터: (2763, 48)
테스트 데이터: (691, 48)


In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 모델 학습
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 예측 및 평가
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE (평균 절대 오차): {mae:.3f}")
print(f"R² 스코어: {r2:.3f}")

MAE (평균 절대 오차): 0.176
R² 스코어: 0.852


In [8]:
import joblib

joblib.dump(model, "price_model.joblib")
print("모델 저장 완료!")

모델 저장 완료!


In [9]:
import os
print(os.listdir())

['data', 'price_model.joblib', 'train_price_model.ipynb']
